# Projeto Isabel — MDT pré-1981 (Colab)
Roda o pipeline do repositório. Não precisa baixar nenhum script: tudo vem do GitHub.

**Repo privado?** Crie um token (GitHub → Settings → Developer settings → Fine-grained tokens, leitura de *Contents*) e guarde no Colab em 🔑 Secrets com o nome `GH_TOKEN`.

In [ ]:
REPO = 'paivaai/isabel-mdt-pre81'   # <- ajuste se o nome for outro
import os
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f'https://{tok+"@" if tok else ""}github.com/{REPO}.git'
if not os.path.exists('/content/isabel'):
    !git clone -q {url} /content/isabel
else:
    !git -C /content/isabel pull -q
%cd /content/isabel
!pip -q install -e .
!apt-get -qq install -y libarchive13 unrar-free > /dev/null 2>&1

## Dados de entrada
Suba `landsat_1979.tif` e `NA_TD_TOTAL.zip` (ou .rar) — vão para `data/`. Se já estiverem no seu Drive, monte-o e copie.

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('data', exist_ok=True)
for nome in files.upload():
    shutil.move(nome, os.path.join('data', nome))
print(os.listdir('data'))

## Rodar
Ajuste `config/projeto.yml` se precisar (datum das curvas, resolução...). Etapas: `aoi landsat drenagem curvas mdt webmap` ou `all`.

In [ ]:
!python -m mdt_pre81 all

## Ver o webmap aqui e baixar os resultados

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/resultados', 'zip', 'outputs')
shutil.make_archive('/content/webmap_site', 'zip', 'docs')
files.download('/content/resultados.zip'); files.download('/content/webmap_site.zip')

## (Opcional) Publicar o webmap no GitHub
Precisa de token com permissão de escrita (secret `GH_TOKEN`). Depois o GitHub Pages atualiza sozinho.

In [ ]:
!git config user.email 'colab@example.com' && git config user.name 'Colab'
!git add docs && git commit -m 'Atualiza webmap' && git push